# Geometry and Motion: Projection, Epipolar Lines and Structure from Motion

Prepared for Fall 2026. Reference: the notes for Lecture 16 (Geometry and Motion: Stereo, Structure from Motion). If you find a bug in this notebook, please email the TAs or tell us at office hours.

**Before you start.** Everything is synthetic, computed with numpy under fixed seeds, so you get our numbers. Every figure and number a question asks for must appear in your PDF: to vary a parameter, copy the cell below the question (or loop) rather than re-running a cell above it. Question 9 asks for eleven figures (six of them with several panels) and about twenty numbers or short tables; each bullet says which and needs at most a few lines of code.

## The camera

The camera of the notes is a pinhole $\vec o$, right-handed unit axes $\vec a$ (right), $\vec b$ (down), $\vec c$ (the gaze), $\vec a\times\vec b=\vec c$, and a focal length $f$; the imaging plane $(\vec r-\vec o)\cdot\vec c=-f$ lies *behind* the pinhole, so a scene point $\vec r$ projects to

$$u=-f\,\frac{(\vec r-\vec o)\cdot\vec a}{(\vec r-\vec o)\cdot\vec c},\qquad v=-f\,\frac{(\vec r-\vec o)\cdot\vec b}{(\vec r-\vec o)\cdot\vec c}.$$

The minus signs are the inversion of the image; the division by the depth $(\vec r-\vec o)\cdot\vec c$ makes the projection non-linear. World coordinates are $X$ right, $Y$ down, $Z$ forward; `camera()` with its defaults ($\vec o=0$, axes $(1,0,0),(0,1,0),(0,0,1)$, $f=1$) is the untilted camera; `yaw_deg` turns it sideways, `tilt_deg` tilts it down and `roll_deg` rotates it about the gaze (see its docstring). **Display:** a point to the right of the gaze has $u<0$ and one below it $v<0$, so the plots let $u$ increase to the left and $v$ upward: the image drawn upright.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)

COLOUR = {"edge": "0.35", "extended": "purple", "vanishing": "purple", "fitted": "teal", "horizon": "orange",
          "run": "teal", "minimum": "orange", "truth": "k", "recovered": "orange", "epipole": "purple",
          "perspective": "0.35", "orthographic": "orange"}


def fmt(values, digits: int = 4) -> str:
    """Format numbers with `digits` significant figures, separated by two spaces."""
    return "  ".join(f"{float(v):.{digits}g}" for v in np.atleast_1d(values))


def unit(v):
    """v / |v| (ValueError for the zero vector)."""
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    if n == 0:
        raise ValueError("cannot normalise the zero vector")
    return v / n


# ------------------------------------------------------------------ 1. The camera and its projection

def camera(position=(0.0, 0.0, 0.0), yaw_deg: float = 0.0, tilt_deg: float = 0.0, roll_deg: float = 0.0, f: float = 1.0) -> dict:
    """The camera of the notes: pinhole o, right-handed unit axes a (right), b (down), c (the gaze) with a x b = c,
    focal length f. World coordinates: X right, Y down, Z forward. yaw_deg turns the gaze about the vertical (positive
    to the right), tilt_deg then tilts it down (positive looks down), roll_deg turns the image axes about the gaze;
    yaw = tilt = roll = 0 gives a = (1, 0, 0), b = (0, 1, 0), c = (0, 0, 1). Returns {"o", "a", "b", "c", "f"}."""
    psi, theta, rho = np.radians(yaw_deg), np.radians(tilt_deg), np.radians(roll_deg)
    a0 = np.array([np.cos(psi), 0.0, -np.sin(psi)])
    b0 = np.array([0.0, 1.0, 0.0])
    c0 = np.array([np.sin(psi), 0.0, np.cos(psi)])
    c = np.cos(theta) * c0 + np.sin(theta) * b0            # tilt: rotate the gaze down about a0
    b1 = np.cos(theta) * b0 - np.sin(theta) * c0
    a = np.cos(rho) * a0 + np.sin(rho) * b1                # roll: rotate a and b about the gaze
    b = -np.sin(rho) * a0 + np.cos(rho) * b1
    assert np.allclose(np.cross(a, b), c), "the axes must be right-handed: a x b = c"
    return {"o": np.asarray(position, dtype=float), "a": a, "b": b, "c": c, "f": float(f)}


def project(points, cam: dict):
    """Perspective projection with the imaging plane behind the pinhole:
    u = -f (r - o).a / (r - o).c,  v = -f (r - o).b / (r - o).c  for every point r (array (N, 3)).
    Returns an (N, 2) array of (u, v); a point at or behind the pinhole (depth (r - o).c <= 0) gives nan."""
    r = np.atleast_2d(np.asarray(points, dtype=float)) - cam["o"]
    depth = r @ cam["c"]
    safe = np.where(depth > 0, depth, np.nan)
    with np.errstate(invalid="ignore", divide="ignore"):
        u = -cam["f"] * (r @ cam["a"]) / safe
        v = -cam["f"] * (r @ cam["b"]) / safe
    return np.column_stack([u, v])


def project_scaled_orthographic(points, cam: dict, Z0: float):
    """Scaled orthographic projection: the depth (r - o).c is replaced by the constant Z0, so
    u = -(f / Z0) (r - o).a,  v = -(f / Z0) (r - o).b, linear in r. Returns an (N, 2) array."""
    r = np.atleast_2d(np.asarray(points, dtype=float)) - cam["o"]
    s = cam["f"] / Z0
    return np.column_stack([-s * (r @ cam["a"]), -s * (r @ cam["b"])])


def vanishing_point(direction, cam: dict):
    """The vanishing point of all lines with unit direction r_hat: (u, v) = -f (r_hat.a / r_hat.c, r_hat.b / r_hat.c).
    Returns (nan, nan) when r_hat.c = 0: lines perpendicular to the gaze have their vanishing point at infinity."""
    d = unit(direction)
    dc = d @ cam["c"]
    if abs(dc) < 1e-12:
        return np.array([np.nan, np.nan])
    return -cam["f"] * np.array([d @ cam["a"], d @ cam["b"]]) / dc


def fit_vanishing_point(segments_2d):
    """Least-squares intersection of the image lines through the projected segments [(p, q), ...], p and q (u, v) pairs:
    the point (u, v) minimising sum_i (a_i u + b_i v + c_i)^2, where a_i u + b_i v + c_i = 0 is line i with unit
    normal (a_i, b_i), from the normal equations. Returns (nan, nan) when the lines are parallel (all normals agree:
    the point is at infinity) or fewer than two usable lines are given."""
    A, rhs, n_lines = np.zeros((2, 2)), np.zeros(2), 0
    for p, q in segments_2d:
        p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
        if not (np.all(np.isfinite(p)) and np.all(np.isfinite(q))):
            continue
        d = q - p
        n = np.array([-d[1], d[0]])
        length = np.linalg.norm(n)
        if length < 1e-12:
            continue
        n /= length
        A += np.outer(n, n)
        rhs += (n @ p) * n              # c_i = -(n . p); the normal equations are (sum n n^T) x = -sum c_i n
        n_lines += 1
    w = np.linalg.eigvalsh(A)
    if n_lines < 2 or w[0] <= 1e-9 * max(w[1], 1e-300):
        return np.array([np.nan, np.nan])
    return np.linalg.solve(A, rhs)


# ------------------------------------------------------------------ The wire-frame scene

def wireframe_scene(table_angles=(20.0, 50.0), h: float = 1.5, table_positions=((-1.5, 6.0), (1.8, 9.0)),
                    table_side: float = 1.2, table_height: float = 0.75, box_centre_xz=(0.3, 5.0), box_size=(1.0, 0.6, 0.8)):
    """A wire-frame scene for a camera at the origin, h metres above a ground plane (the plane Y = h; Y points down):
    ground lines every metre (X in [-6, 6], Z in [2, 30]); two square tables of side table_side, top table_height above
    the ground, four vertical legs, turned by table_angles (degrees) about the vertical at table_positions (X, Z); an
    axis-aligned box of box_size (X, Y, Z extents) standing on the ground at box_centre_xz.
    Returns a dict family -> {"direction": unit vector, "segments": [(p, q, group), ...]}: each family is one set of
    parallel edges in space (the ground, the box and the tables), group names the object a segment belongs to."""
    families = {}

    def add(name, direction, p, q, group):
        fam = families.setdefault(name, {"direction": unit(direction), "segments": []})
        fam["segments"].append((np.asarray(p, dtype=float), np.asarray(q, dtype=float), group))

    for x in range(-6, 7):
        add("ground and box edges along Z", (0, 0, 1), (x, h, 2.0), (x, h, 30.0), "ground")
    for z in range(2, 31):
        add("ground and box edges along X", (1, 0, 0), (-6.0, h, z), (6.0, h, z), "ground")
    for i, (angle, (tx, tz)) in enumerate(zip(table_angles, table_positions), start=1):
        phi = np.radians(angle)
        d1, d2 = np.array([np.cos(phi), 0.0, np.sin(phi)]), np.array([-np.sin(phi), 0.0, np.cos(phi)])
        centre = np.array([tx, h - table_height, tz])
        half = table_side / 2
        for sign in (-1, 1):
            add(f"table {i} edges at {angle:g} deg", d1, centre - half * d1 + sign * half * d2, centre + half * d1 + sign * half * d2, f"table {i}")
            add(f"table {i} edges at {angle + 90:g} deg", d2, centre - half * d2 + sign * half * d1, centre + half * d2 + sign * half * d1, f"table {i}")
        for s1 in (-1, 1):
            for s2 in (-1, 1):
                top = centre + s1 * half * d1 + s2 * half * d2
                add("vertical edges (legs and box)", (0, 1, 0), top, top + np.array([0.0, table_height, 0.0]), f"table {i}")
    bx, bz = box_centre_xz
    sx, sy, sz = box_size
    xs, ys, zs = (bx - sx / 2, bx + sx / 2), (h - sy, h), (bz - sz / 2, bz + sz / 2)
    for y in ys:
        for z in zs:
            add("ground and box edges along X", (1, 0, 0), (xs[0], y, z), (xs[1], y, z), "box")
    for x in xs:
        for y in ys:
            add("ground and box edges along Z", (0, 0, 1), (x, y, zs[0]), (x, y, zs[1]), "box")
    for x in xs:
        for z in zs:
            add("vertical edges (legs and box)", (0, 1, 0), (x, ys[0], z), (x, ys[1], z), "box")
    return families


def horizon_points(cam: dict):
    """Two points of the horizon: the vanishing points of two horizontal directions 30 degrees either side of the
    gaze's horizontal direction (never perpendicular to the gaze unless it is vertical)."""
    cx, cz = cam["c"][0], cam["c"][2]
    if abs(cx) + abs(cz) < 1e-12:
        return None
    phi = np.arctan2(cx, cz)
    pts = [vanishing_point((np.cos(phi + s), 0.0, np.sin(phi + s)), cam) for s in (-np.pi / 6, np.pi / 6)]
    return pts if all(np.all(np.isfinite(p)) for p in pts) else None


def draw_scene(families: dict, cam: dict, title: str = ""):
    """Two panels: the projected wire-frame (dark grey) with the horizon (orange, dashed), and the same drawing widened
    to show the extended edges of the tables and the box (purple, dashed) running to their vanishing points, the
    analytic vanishing points (purple crosses) and the fitted ones (teal circles). u increases to the left so that the
    image is upright. Returns a dict family -> (analytic (u, v), fitted (u, v)) (nan, nan = at infinity)."""
    fig, axes = plt.subplots(2, 1, figsize=(10, 9), gridspec_kw={"height_ratios": [1, 1.9]})
    table, drawn, scene_pts = {}, [], []
    for name, fam in families.items():
        seg2d = [(project(p, cam)[0], project(q, cam)[0]) for p, q, _ in fam["segments"]]
        groups = [g for _, _, g in fam["segments"]]
        vp_a, vp_f = vanishing_point(fam["direction"], cam), fit_vanishing_point(seg2d)
        table[name] = (vp_a, vp_f)
        for (p, q), g in zip(seg2d, groups):
            scene_pts += [p, q]
            for ax in axes:
                ax.plot([p[0], q[0]], [p[1], q[1]], color=COLOUR["edge"], lw=0.8 if g == "ground" else 1.3)
            if g != "ground" and np.all(np.isfinite(vp_a)):
                near = p if np.hypot(*(p - vp_a)) < np.hypot(*(q - vp_a)) else q
                axes[1].plot([near[0], vp_a[0]], [near[1], vp_a[1]], color=COLOUR["extended"], lw=0.6, ls="--", alpha=0.7)
        if np.all(np.isfinite(vp_a)):
            drawn.append(vp_a)
            axes[1].plot(*vp_a, "x", color=COLOUR["vanishing"], ms=9, mew=2, zorder=5)
        if np.all(np.isfinite(vp_f)):
            axes[1].plot(*vp_f, "o", mfc="none", mec=COLOUR["fitted"], ms=11, mew=1.5, zorder=6)
    hp = horizon_points(cam)
    for ax in axes:
        if hp is not None:
            ax.axline(tuple(hp[0]), tuple(hp[1]), color=COLOUR["horizon"], ls="--", lw=1.2, label="horizon")
    axes[1].plot([], [], "x", color=COLOUR["vanishing"], ms=9, mew=2, label="vanishing point (formula)")
    axes[1].plot([], [], "o", mfc="none", mec=COLOUR["fitted"], ms=11, mew=1.5, label="vanishing point (least-squares fit)")
    scene_pts = np.array(scene_pts)
    scene_pts = scene_pts[np.all(np.isfinite(scene_pts), axis=1)]
    if len(scene_pts) == 0:
        raise ValueError("no point of the scene is in front of the camera: turn it back toward the scene")
    for ax, pts, name in zip(axes, (scene_pts, np.vstack([scene_pts] + ([np.array(drawn)] if drawn else []))),
                             ("the image, drawn upright", "with the extended edges and the vanishing points")):
        lo, hi = pts.min(axis=0), pts.max(axis=0)
        pad = 0.06 * (hi - lo).max()
        ax.set_xlim(hi[0] + pad, lo[0] - pad)                 # u increases to the left
        ax.set_ylim(lo[1] - pad, hi[1] + pad)
        ax.set_aspect("equal")
        ax.set_xlabel("$u$ (focal lengths; increasing to the left)")
        ax.set_ylabel("$v$ (focal lengths)")
        ax.set_title(name, fontsize=10)
        ax.legend(fontsize=8, loc="lower left")
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()
    return table


def print_vanishing_points(table: dict):
    """The analytic and the fitted vanishing point of every family, and their distance."""
    print(f"{'family':32s} {'formula (u, v)':>22s} {'least-squares fit (u, v)':>26s}   distance")
    def show(vp):
        if not np.all(np.isfinite(vp)):
            return "at infinity"
        vp = np.round(vp, 4) + 0.0                   # + 0.0 drops the sign of a rounded zero ("-0.0000")
        return f"({vp[0]:.4f}, {vp[1]:.4f})"

    for name, (vp_a, vp_f) in table.items():
        sa, sf = show(vp_a), show(vp_f)
        dist = f"{np.hypot(*(vp_a - vp_f)):.1e}" if np.all(np.isfinite(vp_a)) and np.all(np.isfinite(vp_f)) else "-"
        print(f"{name:32s} {sa:>22s} {sf:>26s}   {dist}")


def box_corners(centre, size):
    """The eight corners of an axis-aligned box with the given centre and (X, Y, Z) extents, array (8, 3)."""
    c, s = np.asarray(centre, dtype=float), np.asarray(size, dtype=float)
    return np.array([c + s * np.array([sx, sy, sz]) / 2 for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)])


def box_approximation_error(Z0s, cam: dict, box_size=(1.0, 0.6, 0.8), x_offset: float = 0.3, h: float = 1.5):
    """The box of box_size standing on the ground (Y = h) at X = x_offset, its centre at depth Z0 in front of the level
    camera `cam`, rendered under perspective and under scaled orthographic projection with Z0 the depth of its centre.
    For each Z0: the maximum over the eight corners of |u_perspective - u_scaled orthographic|, divided by the width
    (range of u) of the scaled orthographic image. Returns the relative errors and delta = half the box's depth extent."""
    errors = []
    for Z0 in np.atleast_1d(Z0s):
        corners = box_corners((x_offset, h - box_size[1] / 2, Z0), box_size)
        persp, so = project(corners, cam), project_scaled_orthographic(corners, cam, Z0)
        errors.append(np.max(np.abs(persp[:, 0] - so[:, 0])) / np.ptp(so[:, 0]))
    return np.array(errors), box_size[2] / 2


def draw_box_approximation(Z0s, errors, delta, cam: dict, show_depths=(3.0, 30.0), box_size=(1.0, 0.6, 0.8), x_offset: float = 0.3, h: float = 1.5):
    """Left panels: the box at the depths show_depths under perspective (grey) and scaled orthographic projection
    (orange, dashed), drawn upright; right panel: the relative error against delta / Z0 on log axes, with 1 % marked.
    The legend of all three panels sits in the right panel, so that it does not cover the box."""
    fig, axes = plt.subplots(1, len(show_depths) + 1, figsize=(4.2 * (len(show_depths) + 1), 3.6))
    edges = [(i, j) for i in range(8) for j in range(i + 1, 8) if bin(i ^ j).count("1") == 1]
    for ax, Z0 in zip(axes[:-1], show_depths):
        corners = box_corners((x_offset, h - box_size[1] / 2, Z0), box_size)
        persp, so = project(corners, cam), project_scaled_orthographic(corners, cam, Z0)
        for i, j in edges:
            ax.plot(persp[[i, j], 0], persp[[i, j], 1], color=COLOUR["perspective"], lw=1.4)
            ax.plot(so[[i, j], 0], so[[i, j], 1], color=COLOUR["orthographic"], lw=1.2, ls="--")
        ax.invert_xaxis()
        ax.set_aspect("equal")
        ax.set_title(f"the box {Z0:g} m away", fontsize=10)
        ax.set_xlabel("$u$ (increasing to the left)")
        ax.set_ylabel("$v$")
    ax = axes[-1]
    ax.plot([], [], color=COLOUR["perspective"], label="perspective (left panels)")
    ax.plot([], [], color=COLOUR["orthographic"], ls="--", label="scaled orthographic (left panels)")
    ax.loglog(delta / np.asarray(Z0s), errors, color=COLOUR["fitted"], label="relative error")
    ax.axhline(0.01, ls="--", color="0.4", lw=1, label="1 %")
    ax.set_xlabel(r"$\delta / Z_0$")
    ax.set_ylabel("max relative error of the box's image")
    ax.set_title("scaled orthographic vs perspective", fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## Vanishing points, the horizon and scaled orthographic projection

Parallel lines $\vec r_0+\lambda\hat{\vec r}$ share one unit direction; with the base point slid to $\vec r_0\cdot\vec c=0$ the image of the point at $\lambda$ is a constant plus a term in $1/\lambda$, so as $\lambda\to\infty$ all lines with direction $\hat{\vec r}$ meet at the **vanishing point**

$$(u,v)\to-f\Big(\frac{\hat{\vec r}\cdot\vec a}{\hat{\vec r}\cdot\vec c},\ \frac{\hat{\vec r}\cdot\vec b}{\hat{\vec r}\cdot\vec c}\Big),$$

independent of $\vec r_0$; when $\hat{\vec r}\cdot\vec c=0$ it is at infinity and the images stay parallel. The **horizon** holds the vanishing points of all horizontal directions; square tables on a flat floor seen by a tilted camera give two vanishing points each on the horizon and one more for all the legs, $2\times(\text{tables})+1$. `fit_vanishing_point` intersects the projected lines of one family by least squares. **Scaled orthographic projection** replaces the depth $\vec r\cdot\vec c$ by a constant $Z_0$, $u=-(f/Z_0)\,\vec r\cdot\vec a$, $v=-(f/Z_0)\,\vec r\cdot\vec b$: linear, and accurate when the depth is large and varies little over the object.

**The cell:** the scene with its vanishing points by formula and by fit (change the angles if you like); the legs' vanishing point against the tilt (`tilts`, `v_legs`, `rc_legs` $=\hat{\vec r}\cdot\vec c$); and the box pushed from 3 m to 60 m, with the largest difference in $u$ between its perspective and its scaled orthographic image, divided by the image width, against $\delta/Z_0$ ($\delta$ half its depth extent; `Z0s`, `errors`, `delta`).

In [ ]:
h, tilt, f = 1.5, 25.0, 1.0            # camera height (m), tilt down (degrees), focal length
cam = camera(position=(0.0, 0.0, 0.0), yaw_deg=0.0, tilt_deg=tilt, f=f)
families = wireframe_scene(table_angles=(20.0, 50.0), h=h)       # tables turned by 20 and 50 degrees (see the docstring for the rest)
print(f"camera {h} m above the ground, tilted {tilt:g} degrees down, f = {f:g}; {len(families)} families of parallel edges "
      f"(two tables give 2 x 2 + 1 = 5 vanishing points; the ground and the box add the edges along X and along Z)")
vp_table = draw_scene(families, cam, title=f"the scene seen by the camera tilted {tilt:g} degrees down (purple: extended edges and vanishing points; orange: horizon)")
print_vanishing_points(vp_table)

# the legs' vanishing point as the camera tilt goes from 30 degrees down to 0.5
vertical = np.array([0.0, 1.0, 0.0])
tilts = np.arange(30.0, 0.45, -0.1)
v_legs = np.array([vanishing_point(vertical, camera(tilt_deg=t, f=f))[1] for t in tilts])
rc_legs = np.array([vertical @ camera(tilt_deg=t, f=f)["c"] for t in tilts])          # r_hat . c for the vertical direction
fig, ax = plt.subplots(figsize=(6.2, 3.3))
ax.semilogy(tilts, np.abs(v_legs), color=COLOUR["vanishing"], lw=1.8, label="$|v|$ of the legs' vanishing point")
ax.axhline(10 * f, ls="--", color="0.4", lw=1, label="$|v| = 10 f$")
ax.set_xlim(tilts[0], tilts[-1])
ax.set_xlabel("tilt of the camera (degrees down)")
ax.set_ylabel("$|v|$ (focal lengths)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
print("tilt (deg)    v of the legs' vanishing point    r_hat . c")
for t in (30.0, 20.0, 10.0, 8.0, 6.0, 5.0, 4.0, 2.0, 1.0, 0.5):
    i = int(np.argmin(np.abs(tilts - t)))
    print(f"{tilts[i]:8.1f}    {v_legs[i]:26.4f}    {rc_legs[i]:9.4f}")

# the box (1 x 0.6 x 0.8 m, on the ground) pushed from 3 m to 60 m in front of a level camera: perspective against scaled
# orthographic projection with Z0 the depth of the box's centre
level = camera(f=f)
Z0s = np.geomspace(3.0, 60.0, 300)
errors, delta = box_approximation_error(Z0s, level)
print(f"box 1 x 0.6 x 0.8 m on the ground, delta = {delta:g} m; relative error {100 * errors[0]:.1f} % at Z0 = 3 m, "
      f"{100 * errors[-1]:.2f} % at Z0 = 60 m")
draw_box_approximation(Z0s, errors, delta, level)

### Question 9.1 (3 points)

- Show the rendered scene with the vanishing points of every edge family (crosses: formula; circles: fit) and the horizon, and the table of both sets of coordinates. **(1 point)**
- Show the plot of the legs' vanishing point against the tilt: at which tilt does it leave a frame of ten focal lengths ($|v|>10f$), and what is $\hat{\vec r}\cdot\vec c$ for the vertical direction there? **(1 point)**
- Show the error curve for the box and report the smallest depth ratio $Z_0/\delta$ at which the error falls below 1 %. **(1 point)**

In [ ]:
# Your code for Question 9.1, e.g.
# i = np.argmax(np.abs(v_legs) > 10 * f)                      # tilts run from 30 down to 0.5
# print(tilts[i], v_legs[i], rc_legs[i])
# j = np.argmax(errors < 0.01)
# print(Z0s[j], Z0s[j] / delta)

**Your answers for Question 9.1:**

## Epipolar lines from $E$ and $F$

In this section the image planes are *in front* of the pinholes with $f=1$, so there is no minus sign: a point $\vec X=(X_1,X_2,X_3)$ in the first camera's coordinates has the homogeneous image $\vec u=(u,v,1)=\vec X/X_3$; in the second camera, whose centre is at $\vec t$, $\vec X'=R(\vec X-\vec t)$ and $\vec u'=\vec X'/X_3'$. The essential matrix

$$E=R\,[\vec t]_\times,\qquad([\vec t]_\times)_{ij}=\sum_k\epsilon_{ijk}t_k,\qquad[\vec t]_\times=\begin{pmatrix}0&t_3&-t_2\\-t_3&0&t_1\\t_2&-t_1&0\end{pmatrix},\qquad[\vec t]_\times\vec X=\vec X\times\vec t,$$

gives the constraint $\vec u'^TE\vec u=0$ on corresponding points. For a fixed $\vec u$, $\vec l=E\vec u$ is its epipolar line in the second image, $l_1u'+l_2v'+l_3=0$. Since $E\vec t=0$, the first image's epipole (the image of the second camera's centre) is $\vec t/t_3$; the second image's epipole $\vec e'$ has $\vec e'^TE=0$, and every line $\vec l$ passes through it. Two parallel cameras a baseline $B$ apart along $\vec a$ see a point at depth $Z$ with the disparity $u-u'=fB/Z$. In pixels $\vec p=K\vec u$ with the calibration matrix $K$, and $\vec p'^TF\vec p=0$ with the fundamental matrix $F=K'^{-T}EK^{-1}$, of rank two.

**The cell:** 200 random points; case A, parallel cameras with $\vec t=(0.3,0,0)$; case B, both cameras turned 12° inward, with the epipole $\vec e'$ marked and six lines $\vec l$ drawn in the second image.

In [ ]:
# ------------------------------------------------------------------ 2. Two cameras: E, epipolar lines, F

def cross_matrix(t):
    """Alan's [t]_x: ([t]_x)_ij = sum_k eps_ijk t_k = [[0, t3, -t2], [-t3, 0, t1], [t2, -t1, 0]], so that [t]_x X = X x t."""
    t1, t2, t3 = np.asarray(t, dtype=float)
    return np.array([[0.0, t3, -t2], [-t3, 0.0, t1], [t2, -t1, 0.0]])


def essential(R, t):
    """E = R [t]_x for cameras related by X' = R (X - t)."""
    return np.asarray(R, dtype=float) @ cross_matrix(t)


def rotation_rows(cam: dict):
    """The 3 x 3 matrix with rows a, b, c: it maps world directions to the camera's coordinates."""
    return np.vstack([cam["a"], cam["b"], cam["c"]])


def camera_coordinates(points, cam: dict):
    """X = R (P - o): world points (N, 3) in the camera's coordinates (rows a, b, c of R); X_3 is the depth."""
    return (np.atleast_2d(np.asarray(points, dtype=float)) - cam["o"]) @ rotation_rows(cam).T


def relative_pose(cam1: dict, cam2: dict):
    """R and t with X' = R (X - t) between the coordinates of two cameras placed in the world: R = R2 R1^T and
    t = R1 (o2 - o1), the second camera's centre in first-camera coordinates."""
    R1, R2 = rotation_rows(cam1), rotation_rows(cam2)
    return R2 @ R1.T, R1 @ (cam2["o"] - cam1["o"])


def homogeneous(X):
    """u = (u, v, 1) = X / X_3 for every row of X (N, 3), the image with f = 1 and the plane in front; depth <= 0 gives nan."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    depth = np.where(X[:, 2] > 0, X[:, 2], np.nan)
    with np.errstate(invalid="ignore", divide="ignore"):
        return X / depth[:, None]


def epipolar_line(E, u):
    """l = E u, the epipolar line l_1 u' + l_2 v' + l_3 = 0 in the second image of the first-image point(s) u ((3,) or (N, 3))."""
    return (np.asarray(E, dtype=float) @ np.atleast_2d(u).T).T


def epipoles(E, t):
    """The two epipoles, scaled to a third coordinate of 1: the first image's e = t / t_3 (E e = 0) and the second
    image's e' with e'^T E = 0 (from numpy.linalg.svd). An epipole whose third coordinate is 0 cannot be scaled and
    is returned as (nan, nan, nan). Returns (e, e')."""
    t = np.asarray(t, dtype=float)
    e = t / t[2] if abs(t[2]) > 1e-9 * np.linalg.norm(t) else np.full(3, np.nan)
    left = np.linalg.svd(np.asarray(E, dtype=float))[0][:, 2]
    e2 = left / left[2] if abs(left[2]) > 1e-9 else np.full(3, np.nan)
    return e, e2


def describe_epipole(e) -> str:
    return "at infinity" if not np.all(np.isfinite(e)) else f"({e[0]:.4f}, {e[1]:.4f})"


def calibration_matrix(fx: float, fy: float, cx: float, cy: float):
    """K = [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]: focal lengths in pixels and the image centre."""
    return np.array([[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]])


def pixels(u, K):
    """p = K u for every row of u (N, 3)."""
    return (np.asarray(K, dtype=float) @ np.atleast_2d(u).T).T


def fundamental(E, K1, K2):
    """F = K2^{-T} E K1^{-1}, so that p'^T F p = 0 for the pixel coordinates p = K1 u, p' = K2 u'."""
    return np.linalg.inv(K2).T @ np.asarray(E, dtype=float) @ np.linalg.inv(K1)


def epipolar_check(E, u, u2):
    """|u'^T E u| for every pair of rows (u in the first image, u2 in the second)."""
    return np.abs(np.einsum("ni,ij,nj->n", np.atleast_2d(u2), np.asarray(E, dtype=float), np.atleast_2d(u)))


def draw_epipolar(E, points_world, cam1: dict, cam2: dict, title: str, n_lines: int = 6, frame: float = 1.0, epipole=None):
    """Two panels, the first and the second image (f = 1, planes in front; v increases downward): all points as small
    grey dots, the first n_lines points in colours, and in the second image their epipolar lines l = E u. If `epipole`
    (a 3-vector with third coordinate 1) is given it is marked with a purple star and the second panel is widened to
    show it. The frame (the square [-frame, frame]^2) is drawn in grey."""
    u1, u2 = homogeneous(camera_coordinates(points_world, cam1)), homogeneous(camera_coordinates(points_world, cam2))
    lines = epipolar_line(E, u1[:n_lines])
    colours = [plt.cm.tab10(i % 10) for i in range(n_lines)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.6 if epipole is not None else 1]})
    for ax, u, name in zip(axes, (u1, u2), ("first image", "second image")):
        ax.plot(u[:, 0], u[:, 1], ".", color="0.6", ms=3, zorder=1)
        for i, col in enumerate(colours):
            ax.plot(u[i, 0], u[i, 1], "o", color=col, ms=6, zorder=4)
        ax.add_patch(plt.Rectangle((-frame, -frame), 2 * frame, 2 * frame, fill=False, ec="0.5", lw=1))
        ax.set_xlabel("$u$")
        ax.set_ylabel("$v$ (downward)")
        ax.set_title(name, fontsize=10)
        ax.set_aspect("equal")
    for (l1, l2, l3), col in zip(lines, colours):
        if abs(l2) > abs(l1):
            p1, p2 = (-6.0, -(l1 * -6.0 + l3) / l2), (6.0, -(l1 * 6.0 + l3) / l2)
        else:
            p1, p2 = (-(l2 * -6.0 + l3) / l1, -6.0), (-(l2 * 6.0 + l3) / l1, 6.0)
        axes[1].axline(p1, p2, color=col, lw=1, zorder=2)
    xlim = [-frame * 1.1, frame * 1.1]
    if epipole is not None and np.all(np.isfinite(epipole)):
        axes[1].plot(epipole[0], epipole[1], "*", color=COLOUR["epipole"], ms=14, zorder=5, label="epipole $e'$")
        axes[1].legend(fontsize=8, loc="upper right")
        xlim = [min(xlim[0], epipole[0] - 0.3), max(xlim[1], epipole[0] + 0.3)]
    axes[0].set_xlim(-frame * 1.1, frame * 1.1)
    axes[1].set_xlim(*xlim)
    for ax in axes:
        ax.set_ylim(frame * 1.1, -frame * 1.1)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()


def plot_disparity(depth, disparity, B: float, f: float = 1.0):
    """The disparity u - u' of every point against its depth Z, with the curve f B / Z drawn through it."""
    fig, ax = plt.subplots(figsize=(5.5, 3.3))
    ax.plot(depth, disparity, "o", ms=3, color=COLOUR["fitted"], label="the 200 points")
    Z = np.linspace(np.min(depth), np.max(depth), 200)
    ax.plot(Z, f * B / Z, color=COLOUR["horizon"], lw=1.5, label="$f B / Z$")
    ax.set_xlabel("depth $Z$ (m)")
    ax.set_ylabel("disparity $u - u'$ (focal lengths)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

In [ ]:
rng = np.random.default_rng(0)
points = np.column_stack([rng.uniform(-2, 2, 200), rng.uniform(-2, 2, 200), rng.uniform(3, 8, 200)])   # world coordinates: X, Y in [-2, 2], Z in [3, 8]

# Case A: parallel cameras, the second displaced by 0.3 m along a; relative_pose gives R, t for two cameras placed in the world
cam1_A, cam2_A = camera(), camera(position=(0.3, 0.0, 0.0))
R_A, t_A = relative_pose(cam1_A, cam2_A)
assert np.allclose(cross_matrix(t_A) @ points[0], np.cross(points[0], t_A)), "[t]_x X must equal X x t (Alan's sign)"
E_A = essential(R_A, t_A)
u_A, u2_A = homogeneous(camera_coordinates(points, cam1_A)), homogeneous(camera_coordinates(points, cam2_A))
depth_A = camera_coordinates(points, cam1_A)[:, 2]
e_A, e2_A = epipoles(E_A, t_A)
print(f"case A: R = I: {np.allclose(R_A, np.eye(3))}, t = {t_A}")
print(f"largest |u'^T E u| over the 200 points: {epipolar_check(E_A, u_A, u2_A).max():.2e}")
if np.all(np.isfinite(e_A)) and np.all(np.isfinite(e2_A)):
    print(f"epipoles: first image {describe_epipole(e_A)}, second image {describe_epipole(e2_A)}")
draw_epipolar(E_A, points, cam1_A, cam2_A, "case A: parallel cameras, t = (0.3, 0, 0)")

# Case B: both cameras turned 12 degrees inward about the vertical
cam1_B, cam2_B = camera(position=(-0.15, 0.0, 0.0), yaw_deg=12.0), camera(position=(0.15, 0.0, 0.0), yaw_deg=-12.0)
R_B, t_B = relative_pose(cam1_B, cam2_B)
E_B = essential(R_B, t_B)
u_B, u2_B = homogeneous(camera_coordinates(points, cam1_B)), homogeneous(camera_coordinates(points, cam2_B))
e_B, e2_B = epipoles(E_B, t_B)
lines_B = epipolar_line(E_B, u_B[:6])
print(f"\ncase B: relative rotation {np.degrees(np.arccos((np.trace(R_B) - 1) / 2)):.1f} degrees, t = {t_B}")
print("E = R [t]_x =")
print(E_B)
print(f"largest |u'^T E u| over the 200 points: {epipolar_check(E_B, u_B, u2_B).max():.2e}")
print(f"epipoles: first image {describe_epipole(e_B)}, second image e' = {describe_epipole(e2_B)}")
print(f"largest |l . e'| over the six lines drawn: {np.abs(lines_B @ e2_B).max():.2e}")
draw_epipolar(E_B, points, cam1_B, cam2_B, "case B: converging cameras, each turned 12 degrees inward", epipole=e2_B)

### Question 9.2 (4 points)

- Show the parallel-camera figure with its six epipolar lines and report the largest $|\vec u'^TE\vec u|$ over the 200 points. **(1 point)**
- Plot the disparity $u-u'$ of the 200 points against their depth with the curve $fB/Z$ through it (`plot_disparity`). Where is the epipole of the parallel pair, and which entry of $\vec t$ puts it there? **(1 point)**
- Show the converging-camera figure with the epipole $\vec e'$ marked and check that the six lines pass through it: report the largest $|\vec l\cdot\vec e'|$. **(1 point)**
- In pixels, for case B with $K=K'$ ($f_x=f_y=800$, $c_x=320$, $c_y=240$): report the largest $|\vec p'^TF\vec p|$ over the 200 points and the singular values of $E$ and of $F$. **(1 point)**

In [ ]:
# Your code for Question 9.2, e.g.
# plot_disparity(depth_A, u_A[:, 0] - u2_A[:, 0], B=0.3, f=1.0)
# K = calibration_matrix(800, 800, 320, 240)
# F_B = fundamental(E_B, K, K)
# p_B, p2_B = pixels(u_B, K), pixels(u2_B, K)
# print(epipolar_check(F_B, p_B, p2_B).max())
# print(fmt(np.linalg.svd(E_B, compute_uv=False)), fmt(np.linalg.svd(F_B, compute_uv=False)))    # fmt: four significant digits (the default array print rounds small values to 0)

**Your answers for Question 9.2:**

## Factorisation: structure from motion under linear projection

$N$ points $\vec X^n$ are seen in $M$ views; view $m$ projects with the rows $\vec K^m,\vec H^m$ of an affine camera, $u^{n,m}=\vec K^m\cdot\vec X^n$, $v^{n,m}=\vec H^m\cdot\vec X^n$, each coordinate measured from its view's centroid, which removes the camera translation. The $M\times N$ matrix $U$ of the $u^{n,m}$ is $KX$, $K$ the $M\times3$ matrix of rows $\vec K^m$ (not the calibration matrix of Question 9.2) and $X$ the $3\times N$ matrix of points: rank at most three. The sum of squared errors $E(K,X)=\sum_{n,m}(u^{n,m}-\vec K^m\cdot\vec X^n)^2$ is bilinear and non-convex, but every local minimum is global and the other critical points are saddle points, where steepest descent stalls on a plateau. The SVD $U=EDF^T$ (singular values $d_1\ge d_2\ge\dots$; the notes reuse the letters $E$ and $F$ for these two orthogonal matrices, so the code says `energy` and `E_svd`) gives the global minimum, of value $\sum_{k>3}d_k^2$. Ambiguity: $K\mapsto KA$, $X\mapsto A^{-1}X$ for any invertible $A$. Stacking the $u$ and $v$ rows into one $2M\times N$ matrix, a single SVD gives rows $\hat K^m,\hat H^m$ and a shape $\hat X$, and the conditions of scaled orthographic projection, $\vec K^m\cdot\vec H^m=0$ and $|\vec K^m|=|\vec H^m|$, are linear in the six entries of $Q=AA^T$:

$$\hat K^mQ\hat H^{mT}=0,\qquad\hat K^mQ\hat K^{mT}=\hat H^mQ\hat H^{mT}.$$

What is left is $A\mapsto AO$ with $O$ orthogonal: a scale and a rotation, possibly with a mirror reflection in depth — orthographic projection cannot tell a shape from its mirror image.

**The cell:** a 36-point barn in $M=10$ scaled orthographic views with 1 % noise; its singular values, the factorisation and the alignment by the best rotation, reflection and scale; steepest descent on the $u$ rows (step 0.01, 3000 steps, 8 small random starts; the notes use 0.008 and 2500 on their house — choose your own).

In [ ]:
# ------------------------------------------------------------------ 3. Factorisation

def barn(seed: int = 0):
    """The 36-point object: a box 2 m wide (X), 1.2 m high (Y, pointing down: the box spans Y in [-1.2, 0]) and 1 m deep
    (Z) with a ridge roof 0.5 m above it; its 10 vertices (8 corners, 2 ridge ends), the 12 midpoints of the box edges,
    the centres of the two roof slopes and 12 interior points at random positions (seed). Returns the points (3, N),
    centred on their centroid, and the 17 edges (pairs of vertex indices) for drawing."""
    hx, hy, hz = 1.0, 1.2, 0.5
    corners = np.array([[sx * hx, yy, sz * hz] for yy in (0.0, -hy) for sx in (-1, 1) for sz in (-1, 1)])
    ridge = np.array([[-hx, -hy - 0.5, 0.0], [hx, -hy - 0.5, 0.0]])
    vertices = np.vstack([corners, ridge])
    edges = [(0, 1), (2, 3), (0, 2), (1, 3), (4, 5), (6, 7), (4, 6), (5, 7), (0, 4), (1, 5), (2, 6), (3, 7),
             (4, 8), (5, 8), (6, 9), (7, 9), (8, 9)]
    midpoints = np.array([(vertices[i] + vertices[j]) / 2 for i, j in edges[:12]])
    slopes = np.array([vertices[[4, 6, 9, 8]].mean(axis=0), vertices[[5, 7, 9, 8]].mean(axis=0)])
    rng = np.random.default_rng(seed)
    interior = np.column_stack([rng.uniform(-hx, hx, 12), rng.uniform(-hy, 0.0, 12), rng.uniform(-hz, hz, 12)])
    points = np.vstack([vertices, midpoints, slopes, interior]).T
    return points - points.mean(axis=1, keepdims=True), edges


def scaled_orthographic_rows(cam: dict):
    """The rows K = -f a and H = -f b of the scaled orthographic camera: an affine camera with K.H = 0 and |K| = |H|."""
    return -cam["f"] * cam["a"], -cam["f"] * cam["b"]


def stack_views(points3d, cams, noise_sd: float = 0.0, seed: int = 0):
    """The 2M x N matrix of the notes: rows 1..M hold u^{n,m} = K^m . X^n, rows M+1..2M hold v^{n,m} = H^m . X^n, with
    Gaussian noise of s.d. noise_sd added to every coordinate and each row then measured from its centroid (its mean
    over the points is subtracted). points3d: (3, N). Returns the (2M, N) array."""
    rng = np.random.default_rng(seed)
    rows_u, rows_v = [], []
    for cam in cams:
        K, H = scaled_orthographic_rows(cam)
        rows_u.append(K @ points3d)
        rows_v.append(H @ points3d)
    W = np.vstack(rows_u + rows_v)
    W = W + rng.normal(0.0, noise_sd, size=W.shape) if noise_sd > 0 else W
    return W - W.mean(axis=1, keepdims=True)


def factor_rank3(W):
    """The SVD W = E D F^T (numpy.linalg.svd) and the rank-three factors with P = Q = D_3^(1/2):
    Khat = E_3 D_3^(1/2) (its rows are the Khat^m, then the Hhat^m, for a stacked matrix) and Xhat = D_3^(1/2) F_3^T.
    Returns Khat (rows x 3), Xhat (3 x N) and all the singular values d."""
    E_svd, d, Ft = np.linalg.svd(np.asarray(W, dtype=float), full_matrices=False)
    sq = np.sqrt(d[:3])
    return E_svd[:, :3] * sq, sq[:, None] * Ft[:3], d


def solve_Q(Khat, Hhat):
    """Q = A A^T from the orthogonality constraints, linear in the six entries of the symmetric Q: for every view,
    Khat^m Q Hhat^mT = 0 and Khat^m Q Khat^mT - Hhat^m Q Hhat^mT = 0, plus the normalisation sum_m Khat^m Q Khat^mT = M
    (2M + 1 equations, numpy.linalg.lstsq); then A = V diag(sqrt(w)) from the eigen-decomposition Q = V diag(w) V^T.
    A non-positive eigenvalue (the noise is too large for the constraints) is replaced by its absolute value with a
    printed warning. Returns Q, A."""
    def coefficients(k, h):           # of (Q11, Q12, Q13, Q22, Q23, Q33) in k Q h^T
        return np.array([k[0] * h[0], k[0] * h[1] + k[1] * h[0], k[0] * h[2] + k[2] * h[0], k[1] * h[1], k[1] * h[2] + k[2] * h[1], k[2] * h[2]])
    rows, rhs = [], []
    for k, h in zip(Khat, Hhat):
        rows.append(coefficients(k, h)); rhs.append(0.0)
        rows.append(coefficients(k, k) - coefficients(h, h)); rhs.append(0.0)
    rows.append(sum(coefficients(k, k) for k in Khat)); rhs.append(float(len(Khat)))
    q = np.linalg.lstsq(np.array(rows), np.array(rhs), rcond=None)[0]
    Q = np.array([[q[0], q[1], q[2]], [q[1], q[3], q[4]], [q[2], q[4], q[5]]])
    w, V = np.linalg.eigh(Q)
    if np.any(w <= 0):
        print(f"warning: Q has a non-positive eigenvalue ({w.min():.3g}); its absolute value is used")
        w = np.abs(w)
    return Q, V * np.sqrt(w)


def align(shape, truth, allow_reflection: bool = True):
    """Align `shape` (3 x N) to `truth` (3 x N) by the best rotation, reflection and scale (one SVD of truth shape^T
    after centring both): returns s O shape, moved onto the centroid of truth. With allow_reflection=False only
    rotations are allowed (det O = +1)."""
    S = shape - shape.mean(axis=1, keepdims=True)
    T = truth - truth.mean(axis=1, keepdims=True)
    Wm, sig, Vt = np.linalg.svd(T @ S.T)
    D = np.eye(3)
    if not allow_reflection and np.linalg.det(Wm @ Vt) < 0:
        D[2, 2] = -1.0
    O = Wm @ D @ Vt
    denom = float(np.sum(S * S))
    s = float(np.sum(sig * np.diag(D)) / denom) if denom > 0 else 1.0
    return s * (O @ S) + truth.mean(axis=1, keepdims=True)


def rms_error(shape, truth) -> float:
    """Root-mean-square distance between corresponding points of two 3 x N shapes."""
    return float(np.sqrt(np.mean(np.sum((np.asarray(shape) - np.asarray(truth)) ** 2, axis=0))))


def energy(K, X, U) -> float:
    """E(K, X) = sum_{n,m} (u^{n,m} - K^m . X^n)^2 for the M x N matrix U."""
    return float(np.sum((U - K @ X) ** 2))


def steepest_descent_E(U, step: float = 0.01, n_steps: int = 3000, n_starts: int = 8, start_sd: float = 0.01, seed: int = 0):
    """Steepest descent on E(K, X) with the u rows only (U: M x N) from n_starts small random starts (entries of K
    and X drawn from N(0, start_sd^2)): K <- K - step dE/dK, X <- X - step dE/dX with dE/dK = -2 (U - K X) X^T and
    dE/dX = -2 K^T (U - K X). Returns the energies, array (n_starts, n_steps + 1); a run whose energy overflows stops
    (nan afterwards) with a printed message that the step is too large."""
    U = np.asarray(U, dtype=float)
    M, N = U.shape
    rng = np.random.default_rng(seed)
    energies = np.full((n_starts, n_steps + 1), np.nan)
    for r in range(n_starts):
        K, X = rng.normal(0.0, start_sd, (M, 3)), rng.normal(0.0, start_sd, (3, N))
        for s in range(n_steps + 1):
            residual = U - K @ X
            e = float(np.sum(residual ** 2))
            if not np.isfinite(e) or e > 1e12:
                print(f"run {r + 1}: E overflowed at step {s}; the step {step} is too large for this matrix")
                break
            energies[r, s] = e
            if s == n_steps:
                break
            K, X = K + 2 * step * residual @ X.T, X + 2 * step * K.T @ residual
    return energies


def plateau_level(energies, factor: float = 1.5, bin_width: float = 0.05):
    """For each run, the level at which it spends the most steps before converging: the energies above `factor` times
    the run's final value are binned on a log scale (bins of relative width bin_width) and the median of the fullest
    bin is returned. Returns an array (n_starts,)."""
    levels = []
    for e in np.atleast_2d(energies):
        e = e[np.isfinite(e)]
        if len(e) == 0:                                 # a run that overflowed at its first step
            levels.append(np.nan)
            continue
        high = e[e > factor * e[-1]]
        if len(high) < 2:
            levels.append(np.nan)
            continue
        bins = np.floor(np.log(high) / bin_width).astype(int)
        values, counts = np.unique(bins, return_counts=True)
        levels.append(float(np.median(high[bins == values[np.argmax(counts)]])))
    return np.array(levels)


def plot_singular_values(curves: dict, title: str, marker_cycle=("o", "s", "^", "v", "D")):
    """Singular values (log axis) against their index for several named matrices."""
    fig, ax = plt.subplots(figsize=(5.8, 3.4))
    for (name, d), mk in zip(curves.items(), marker_cycle):
        d = np.asarray(d, dtype=float)
        ax.semilogy(np.arange(1, len(d) + 1), np.maximum(d, 1e-18), mk, ms=5, ls="none", label=name)
    ax.set_xlabel("index $k$")
    ax.set_ylabel("singular value $d_k$")
    ax.set_xticks(np.arange(1, max(len(np.atleast_1d(d)) for d in curves.values()) + 1))
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def plot_shapes(truth, shapes: dict, edges, size: float):
    """Two rows of panels, a front view (X, Y; Y downward) and a top view (X, Z), one column per recovered shape in
    `shapes` (name -> 3 x N array, already aligned): the true shape as black dots with its edges in grey, the recovered
    points as orange circles; the RMS error as a percentage of `size` is written in each title."""
    n = len(shapes)
    fig, axes = plt.subplots(2, n, figsize=(4.3 * n, 7.2), squeeze=False)
    for col, (name, shape) in enumerate(shapes.items()):
        err = 100 * rms_error(shape, truth) / size
        for row, (i, j, yl) in enumerate(((0, 1, "$Y$ (downward)"), (0, 2, "$Z$"))):
            ax = axes[row, col]
            for a, b in edges:
                ax.plot(truth[i, [a, b]], truth[j, [a, b]], color="0.75", lw=1, zorder=1)
            ax.plot(truth[i], truth[j], ".", color=COLOUR["truth"], ms=6, zorder=3, label="true shape")
            ax.plot(shape[i], shape[j], "o", mfc="none", mec=COLOUR["recovered"], ms=8, mew=1.5, zorder=4, label="recovered")
            ax.set_aspect("equal")
            ax.set_xlabel("$X$")
            ax.set_ylabel(yl)
            if row == 0:
                ax.invert_yaxis()
            ax.set_title(f"{name}: {'front' if row == 0 else 'top'} view, RMS error {err:.2f} % of the size", fontsize=9)
            if row == 0 and col == 0:
                ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    plt.show()


def plot_descent(energies, svd_minimum: float, step: float):
    """E(K, X) against the steepest-descent step (log axis) for every run (teal), with the SVD minimum sum_(k>3) d_k^2
    as a dashed orange line."""
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for e in np.atleast_2d(energies):
        ax.semilogy(np.arange(len(e)), e, color=COLOUR["run"], lw=1, alpha=0.8)
    ax.axhline(svd_minimum, ls="--", color=COLOUR["minimum"], lw=1.5, label=r"SVD minimum $\sum_{k>3} d_k^2$")
    ax.plot([], [], color=COLOUR["run"], label=f"{len(np.atleast_2d(energies))} random starts")
    ax.set_xlabel(f"steepest-descent step (step size {step:g})")
    ax.set_ylabel("$E(K, X)$")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

In [ ]:
rng = np.random.default_rng(0)
X_true, barn_edges = barn(seed=0)
M, N = 10, X_true.shape[1]
size = float(np.ptp(X_true, axis=1).max())
# the views turn about the vertical from -50 to 50 degrees, look down from 10 to 40 degrees, are rolled by a random angle
# of up to 5 degrees and have scales f_m in [0.9, 1.1]; K^m = -f_m a_m, H^m = -f_m b_m
azimuths, elevations = np.linspace(-50, 50, M), np.linspace(10, 40, M)
focal, rolls = rng.uniform(0.9, 1.1, M), rng.uniform(-5, 5, M)
cams = [camera(yaw_deg=-az, tilt_deg=el, roll_deg=ro, f=fm) for az, el, ro, fm in zip(azimuths, elevations, rolls, focal)]
print(f"N = {N} points, M = {M} views; object size {size:g} m, so 1 % noise has s.d. {0.01 * size:g}")

W_clean = stack_views(X_true, cams, noise_sd=0.0)
W_noisy = stack_views(X_true, cams, noise_sd=0.01 * size, seed=1)
d_clean, d_noisy = np.linalg.svd(W_clean, compute_uv=False), np.linalg.svd(W_noisy, compute_uv=False)
print(f"singular values of the {2 * M} x {N} matrix, no noise:  {fmt(d_clean, 3)}")
print(f"singular values of the {2 * M} x {N} matrix, 1 % noise: {fmt(d_noisy, 3)}")
plot_singular_values({"no noise": d_clean, "1 % noise": d_noisy}, f"singular values of the {2 * M} x {N} matrix")

# the factorisation of the noisy matrix: the SVD with P = Q = D_3^(1/2), then Q = A A^T from the orthogonality constraints
Khat_full, Xhat, d = factor_rank3(W_noisy)
Khat, Hhat = Khat_full[:M], Khat_full[M:]
Q, A = solve_Q(Khat, Hhat)
X_svd = Xhat                                   # the SVD shape with A = I
X_rec = np.linalg.solve(A, Xhat)               # after the constraints: X = A^-1 Xhat
K_rec, H_rec = Khat @ A, Hhat @ A              # K^m = Khat^m A, H^m = Hhat^m A
print(f"\nconstraints on the recovered rows: max |K^m . H^m| = {np.max(np.abs(np.sum(K_rec * H_rec, axis=1))):.2e}, "
      f"max ||K^m| - |H^m|| = {np.max(np.abs(np.linalg.norm(K_rec, axis=1) - np.linalg.norm(H_rec, axis=1))):.2e}")
print(f"recovered |K^m| divided by the true f_m: {fmt(np.linalg.norm(K_rec, axis=1) / focal, 3)} (one common scale)")
aligned = {"SVD with A = I": align(X_svd, X_true), "after the constraints": align(X_rec, X_true)}
for name, shape in aligned.items():
    print(f"RMS error of the shape '{name}', aligned by the best rotation, reflection and scale: "
          f"{rms_error(shape, X_true):.4f} m = {100 * rms_error(shape, X_true) / size:.2f} % of the size")
plot_shapes(X_true, aligned, barn_edges, size)

# steepest descent on E(K, X) with the u rows only
U = W_noisy[:M]
step, n_steps = 0.01, 3000
energies = steepest_descent_E(U, step=step, n_steps=n_steps, n_starts=8, seed=0)
d_u = np.linalg.svd(U, compute_uv=False)
svd_minimum = float(np.sum(d_u[3:] ** 2))
print(f"\nsingular values of the {M} x {N} matrix U of the u coordinates: {fmt(d_u, 3)}")
print(f"SVD minimum sum_(k>3) d_k^2 = {svd_minimum:.4f};  d_3^2 = {d_u[2] ** 2:.4f};  d_3^2 + minimum = {d_u[2] ** 2 + svd_minimum:.4f}")
print(f"final E of the {len(energies)} runs after {n_steps} steps: {fmt(energies[:, -1], 4)}")
print(f"level at which each run spends the most steps before converging: {fmt(plateau_level(energies), 4)}")
plot_descent(energies, svd_minimum, step)

### Question 9.3 (5 points)

- Show the singular values of the $2M\times N$ matrix without noise and with 1 % noise on a log axis: how many are large in each case? **(1 point)**
- Plot the fourth singular value against the noise s.d. (0, 0.5, 1, 2 and 5 % of the object size). **(1 point)**
- Show the two overlays of the recovered shape on the true one — the SVD shape with $A=I$ and the shape after the orthogonality constraints — with the root-mean-square (RMS) error of each. **(1 point)**
- Show the steepest-descent plot and report the plateau value, compared with $d_3^2$ plus the SVD minimum $\sum_{k>3}d_k^2$. **(1 point)**
- The mirror image: replace $A$ by $A\,\mathrm{diag}(1,1,-1)$ and report the largest change in any projected coordinate; then align the recovered shape and its mirror image to the true one with rotations only (`allow_reflection=False`) and report both RMS errors. **(1 point)**

In [ ]:
# Your code for Question 9.3, e.g.
# d4 = [np.linalg.svd(stack_views(X_true, cams, noise_sd=p / 100 * size, seed=1), compute_uv=False)[3] for p in (0, 0.5, 1, 2, 5)]
# plt.plot((0, 0.5, 1, 2, 5), d4, "o-"); plt.xlabel("noise (% of the object size)"); plt.ylabel("$d_4$"); plt.show()
# A_mirror = A @ np.diag([1.0, 1.0, -1.0])
# X_mirror = np.linalg.solve(A_mirror, Xhat)
# print(np.abs(Khat_full @ A_mirror @ X_mirror - Khat_full @ A @ X_rec).max())
# for shape in (X_rec, X_mirror):
#     print(rms_error(align(shape, X_true, allow_reflection=False), X_true) / size)

**Your answers for Question 9.3:**

## Non-rigid shapes

A non-rigid object is a sum of $d$ basis shapes with coefficients that change with the frame, $\vec X^{n,m}=\sum_{i=1}^d\alpha^m_i\vec b^n_i$, so $u^{n,m}=\sum_i\alpha^m_i\,\vec K^m\cdot\vec b^n_i$ and $U=\tilde K\tilde B$ with $3d$ columns and rows: rank at most $3d$. Any invertible $d\times d$ matrix $G$ replaces the basis by $(\vec b_1,\dots,\vec b_d)G$ and the coefficients by $G^{-1}\alpha^m$; the notes call this a coordinate change.

**The cell:** a 20-point face (`nonrigid_face`) with $\vec b_1$ the neutral face, $\vec b_2$ "open the mouth and drop the jaw" and, for $d=3$, $\vec b_3$ "raise the brows", in 12 frames turning from −35° to 35°; the singular values of $U$ ($M\times N$, the $u$ coordinates) for $d=2$; the coordinate change $G$ with rows $(1,1)$ and $(1,-1)$, and frames 1, 5 and 7 from both bases (dots, circles).

In [ ]:
# ------------------------------------------------------------------ 4. Non-rigid shapes

def face_basis(d: int = 2):
    """A 20-point face (X right, Y down, Z depth; the nose tip is nearest the camera) and its basis shapes:
    b_1 the neutral face; b_2 the displacement that opens the mouth and drops the jaw; b_3 the displacement that raises
    the brows. Returns the list [b_1, ..., b_d] of (3, 20) arrays. Point order: jaw outline (9, left ear to right ear
    through the chin), eyes (4), brows (2), nose (2), mouth (3)."""
    neutral = np.array([
        [-0.70, -0.20, 0.00], [-0.65, 0.20, 0.00], [-0.50, 0.55, -0.05], [-0.25, 0.80, -0.10], [0.00, 0.90, -0.12],
        [0.25, 0.80, -0.10], [0.50, 0.55, -0.05], [0.65, 0.20, 0.00], [0.70, -0.20, 0.00],
        [-0.45, -0.30, -0.05], [-0.15, -0.30, -0.08], [0.15, -0.30, -0.08], [0.45, -0.30, -0.05],
        [-0.30, -0.50, -0.06], [0.30, -0.50, -0.06],
        [0.00, 0.10, -0.30], [0.00, 0.25, -0.15],
        [-0.25, 0.50, -0.10], [0.00, 0.50, -0.14], [0.25, 0.50, -0.10]]).T
    open_mouth = np.zeros((3, 20))                                   # the chin drops and the jaw widens a little
    open_mouth[:, [2, 3, 4, 5, 6]] = np.array([[-0.03, -0.02, 0.0, 0.02, 0.03], [0.10, 0.22, 0.28, 0.22, 0.10], [0.02, 0.05, 0.06, 0.05, 0.02]])
    open_mouth[:, [17, 18, 19]] = np.array([[-0.04, 0.0, 0.04], [0.10, 0.18, 0.10], [0.0, 0.03, 0.0]])
    raise_brows = np.zeros((3, 20))                                  # the brows rise, spread and come forward; the eyes open
    raise_brows[:, [13, 14]] = np.array([[-0.05, 0.05], [-0.22, -0.22], [-0.06, -0.06]])
    raise_brows[:, [9, 10, 11, 12]] = np.array([[-0.02, -0.01, 0.01, 0.02], [-0.06, -0.04, -0.04, -0.06], [-0.01, -0.01, -0.01, -0.01]])
    return [neutral, open_mouth, raise_brows][:d]


def frame_shape(alpha_m, basis):
    """The 3D shape of one frame, sum_i alpha_i b_i (3 x N)."""
    return sum(a * b for a, b in zip(alpha_m, basis))


def nonrigid_face(d: int = 2, n_frames: int = 12, turn_deg: float = 35.0, side_tilt_deg: float = 15.0, f: float = 1.0, seed: int = 0):
    """n_frames frames of the face: frame m has coefficients alpha^m = (1, a_2(m), a_3(m)) with a_2 = (1 - cos(2 pi s))/2
    (the mouth opens and closes) and a_3 = s (the brows rise steadily), s = m/(n_frames - 1), and is seen by a scaled orthographic
    camera (f) turned from -turn_deg to turn_deg about the vertical and rolled by a random angle in [-side_tilt_deg,
    side_tilt_deg] (the head tilted sideways; seed). Returns alpha (M, d), the basis [b_i], the camera rows K (M, 3)
    and H (M, 3), and the M x N matrix U of the u coordinates (each row measured from its centroid)."""
    basis = face_basis(d)
    s = np.arange(n_frames) / (n_frames - 1)
    alpha = np.column_stack([np.ones(n_frames), (1 - np.cos(2 * np.pi * s)) / 2, s])[:, :d]
    rolls = np.random.default_rng(seed).uniform(-side_tilt_deg, side_tilt_deg, n_frames)
    cams = [camera(yaw_deg=yaw, roll_deg=ro, f=f) for yaw, ro in zip(np.linspace(-turn_deg, turn_deg, n_frames), rolls)]
    K = np.array([scaled_orthographic_rows(c)[0] for c in cams])
    H = np.array([scaled_orthographic_rows(c)[1] for c in cams])
    U = np.array([K[m] @ frame_shape(alpha[m], basis) for m in range(n_frames)])
    return alpha, basis, K, H, U - U.mean(axis=1, keepdims=True)


def change_basis(basis, alpha, G):
    """The coordinate change: new basis (b_1, ..., b_d) G, i.e. b'_j = sum_i G_ij b_i, and new coefficients G^-1 alpha^m
    for every frame (alpha: M x d). Returns (new basis, new alpha)."""
    G = np.asarray(G, dtype=float)
    new_basis = [sum(G[i, j] * basis[i] for i in range(len(basis))) for j in range(len(basis))]
    return new_basis, alpha @ np.linalg.inv(G).T


def draw_frames(frames: tuple, alpha, basis, alpha2, basis2, K, H, labels=("basis 1", "basis 2")):
    """The listed frames (1-based) projected by their cameras, drawn upright: dots from the first basis and
    coefficients, circles from the second."""
    fig, axes = plt.subplots(1, len(frames), figsize=(3.4 * len(frames), 3.6))
    for ax, m in zip(np.atleast_1d(axes), frames):
        for (al, ba), style in zip(((alpha, basis), (alpha2, basis2)), ("dots", "circles")):
            shape = frame_shape(al[m - 1], ba)
            u, v = K[m - 1] @ shape, H[m - 1] @ shape
            if style == "dots":
                ax.plot(u, v, ".", color=COLOUR["truth"], ms=7, label=labels[0])
            else:
                ax.plot(u, v, "o", mfc="none", mec=COLOUR["recovered"], ms=9, mew=1.4, label=labels[1])
        ax.invert_xaxis()                           # u increases to the left and v upward: the face drawn upright
        ax.set_aspect("equal")
        ax.set_title(f"frame {m}", fontsize=10)
        ax.set_xlabel("$u$ (increasing to the left)")
        ax.set_ylabel("$v$")
    np.atleast_1d(axes)[0].legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    plt.show()

In [ ]:
alpha, basis, K_face, H_face, U_face = nonrigid_face(d=2, n_frames=12)     # the head is also tilted sideways by a random angle of up to 15 degrees (see the docstring)
sv_face = np.linalg.svd(U_face, compute_uv=False)
print(f"d = 2: singular values of the {U_face.shape[0]} x {U_face.shape[1]} matrix U: {fmt(sv_face, 3)}")
plot_singular_values({"d = 2": sv_face}, "singular values of U for the face")

G = np.array([[1.0, 1.0], [1.0, -1.0]])
basis2, alpha2 = change_basis(basis, alpha, G)
print("coefficients alpha^m (rows: frames) for basis 1, then G^-1 alpha^m for basis 2:")
print(np.column_stack([alpha, alpha2]))
difference = max(np.abs(frame_shape(alpha[m], basis) - frame_shape(alpha2[m], basis2)).max() for m in range(12))
print(f"largest difference between the 3D shapes of the 12 frames from the two bases: {difference:.1e}")
draw_frames((1, 5, 7), alpha, basis, alpha2, basis2, K_face, H_face)

### Question 9.4 (2 points)

- Report the singular values of $U$ for $d=2$ and for $d=3$: how many are above $10^{-6}$ of the largest in each case? **(1 point)**
- Show frames 1, 5 and 7 from both bases and report the largest difference between them. Why is $G$ not an ambiguity that has to be removed? **(1 point)**

In [ ]:
# Your code for Question 9.4, e.g.
# alpha3, basis3, K3, H3, U3 = nonrigid_face(d=3, n_frames=12)
# sv3 = np.linalg.svd(U3, compute_uv=False)
# print(fmt(sv3, 3), np.sum(sv3 > 1e-6 * sv3[0]))

**Your answers for Question 9.4:**